# USGS Data Collection

This notebook fetches the USGS ComCat earthquake catalogue ($M \ge 6.0$, 2000–2025) in a single API call, checking robots.txt and storing the raw CSV without modification.

In [2]:
import os
import requests
import pandas as pd
from urllib.robotparser import RobotFileParser

import os
# If the notebook is running inside the 'notebooks' folder, step back up to the project root
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print(f"Current working directory: {os.getcwd()}")

# 1. Setup Data Folders & Metadata
os.makedirs("raw", exist_ok=True)
USER_AGENT = "DA2402_Project/1.0 (da25b015@smail.iitm.ac.in)"

Current working directory: d:\Pandu\IIT_Madras\DA2402\DA2402_Project_Earthquakes


## 1. Check robots.txt & Terms of Use

In [3]:
robots_url = "https://www.usgs.gov/robots.txt"
resp_robots = requests.get(robots_url, headers={"User-Agent": USER_AGENT})

print(f"Robots.txt Status: {resp_robots.status_code}")
print("--- First 10 lines of robots.txt for the report ---")
print("\n".join(resp_robots.text.splitlines()[:10]))

parser = RobotFileParser()
parser.parse(resp_robots.text.splitlines())
is_allowed = parser.can_fetch(USER_AGENT, "https://earthquake.usgs.gov/fdsnws/event/1/query")
print(f"\nAm I allowed to scrape the API? {is_allowed}")
print("Terms of Use: USGS data is U.S. Government Work (Public Domain / CC0).")

Robots.txt Status: 202
--- First 10 lines of robots.txt for the report ---


Am I allowed to scrape the API? True
Terms of Use: USGS data is U.S. Government Work (Public Domain / CC0).


## 2. End-to-End API Call
The USGS limit is 20,000 events per query. Since we only expect ~3,875 $M \ge 6.0$ earthquakes globally from 2000 to 2025, we can fetch the entire dataset in one call.

In [4]:
api_url = "https://earthquake.usgs.gov/fdsnws/event/1/query"
params = {
    "format": "csv",
    "starttime": "2000-01-01T00:00:00",
    "endtime": "2025-12-31T23:59:59",
    "minmagnitude": 6.0,
    "orderby": "time-asc"
}

print(f"Fetching data from: {api_url}")
resp_api = requests.get(api_url, params=params, headers={"User-Agent": USER_AGENT})
resp_api.raise_for_status()

# Save exactly as it came from the API (Never edit raw files!)
raw_filepath = "raw/usgs_events_2000_2025_2026-10-21.csv"
with open(raw_filepath, "wb") as f:
    f.write(resp_api.content)
    
print(f"Success! Saved raw data to {raw_filepath}")

Fetching data from: https://earthquake.usgs.gov/fdsnws/event/1/query
Success! Saved raw data to raw/usgs_events_2000_2025_2026-10-21.csv


## 3. Raw File Inspection
Displaying shape, dtypes, and the meaning of every kept column

In [5]:
df_raw = pd.read_csv(raw_filepath)
kept_cols = ["id", "time", "latitude", "longitude", "depth", "mag", "magType", "place", "type"]
df_kept = df_raw[kept_cols]

print("--- Raw Data Inspection ---")
print(f"Shape: {df_kept.shape}")
print(f"\nDtypes:\n{df_kept.dtypes}")

column_meanings = {
    "id": "Unique string ID for the earthquake",
    "time": "UTC Timestamp (ISO-8601)",
    "latitude": "WGS84 Latitude (Decimal degrees)",
    "longitude": "WGS84 Longitude (Decimal degrees)",
    "depth": "Focal depth in kilometers",
    "mag": "Reported earthquake magnitude",
    "magType": "Magnitude scale used (e.g., mww, mb)",
    "place": "Unstructured text description of location",
    "type": "Event type (earthquake, nuclear explosion)"
}

print("\nMeaning of kept columns:")
for col, meaning in column_meanings.items():
    print(f" - {col}: {meaning}")

--- Raw Data Inspection ---
Shape: (3875, 9)

Dtypes:
id               str
time             str
latitude     float64
longitude    float64
depth        float64
mag          float64
magType          str
place            str
type             str
dtype: object

Meaning of kept columns:
 - id: Unique string ID for the earthquake
 - time: UTC Timestamp (ISO-8601)
 - latitude: WGS84 Latitude (Decimal degrees)
 - longitude: WGS84 Longitude (Decimal degrees)
 - depth: Focal depth in kilometers
 - mag: Reported earthquake magnitude
 - magType: Magnitude scale used (e.g., mww, mb)
 - place: Unstructured text description of location
 - type: Event type (earthquake, nuclear explosion)
